# Conditional Sample Generation

Generates N samples per conditional model using the chosen diffusers sampler
with classifier-free guidance.

Two models supported (switch via `MODEL` knob):
- `seg2mri`              - Brain MRI conditioned on a segmentation mask
- `novel_segmentation`   - Tumor segmentation map conditioned on size class
                           (cycles through small/medium/large in equal proportion)

Output: `<MODEL>/<SAMPLER>/samples_*/sample_XXXX.nii.gz` + stats.json


In [ ]:
from pathlib import Path

MODEL          = "seg2mri"        # "seg2mri" | "novel_segmentation"
SAMPLER        = "ddim"           # "ddim" | "ddpm" | "dpm_solver++" | "unipc"
N_SAMPLES      = 1024
SAMPLER_STEPS  = 50
GUIDANCE_SCALE = 3.0
SEED_BASE      = 42
IMAGE_SIZE     = 32

CHECKPOINT_ROOT = Path(r"C:\Users\Sven\Desktop\diffusion2\FinishedModels\32")
SEG_DIR         = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\32_seg")   # only used for seg2mri
OUTPUT_ROOT     = Path(rf"{MODEL}/{SAMPLER.upper()}/{IMAGE_SIZE}")

print(f"model={MODEL}  sampler={SAMPLER}  steps={SAMPLER_STEPS}  N={N_SAMPLES}  guidance={GUIDANCE_SCALE}")
print(f"checkpoint root: {CHECKPOINT_ROOT}")
print(f"output:          {OUTPUT_ROOT}")


In [ ]:
import sys
import time
import json
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import nibabel as nib
import torch
from tqdm.auto import tqdm

UNCOND_SRC = Path.cwd().parent / "unconditional" / "src"
COND_SRC   = Path.cwd() / "src"
sys.path.insert(0, str(UNCOND_SRC))
sys.path.insert(0, str(COND_SRC))

from model.unet3d import UNet3D
from sampler.sampler import make_scheduler, sample


# Union Config — needs to cover fields from both conditional model trainings
# so torch.load(weights_only=False) can unpickle whichever was saved.
@dataclass
class Config:
    data_path:     str = ""
    seg_data_path: str = ""
    labels_csv:    str = ""
    output_path:   str = ""
    # model
    image_size:            int = 32
    in_channels:           int = 1
    cond_channels:         int = 0
    model_channels:        int = 96
    channel_mult:          Tuple[int, ...] = (1, 2, 4, 4)
    num_res_blocks:        int = 2
    attention_resolutions: Tuple[int, ...] = (8, 4)
    num_groups:            int = 8
    dropout:               float = 0.0
    # diffusion
    num_timesteps:    int = 250
    beta_schedule:    str = "cosine"
    prediction_type:  str = "v"
    # training
    num_epochs:     int = 300
    batch_size:     int = 10
    learning_rate:  float = 2e-4
    warmup_steps:   int = 500
    ema_decay:      float = 0.9999
    grad_clip:      float = 0.5
    # enhancements
    use_loss_aware_sampling: bool = True
    loss_history_size:       int  = 10
    snr_gamma:               float = 5.0
    # CFG
    cfg_dropout_prob: float = 0.15
    guidance_scale:   float = 3.0
    num_size_classes: int   = 0
    num_seg_classes:  int   = 4
    # device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


sys.modules.setdefault("__main__", sys.modules[__name__])
setattr(sys.modules["__main__"], "Config", Config)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")


In [ ]:
MODELS = {
    "seg2mri": {
        "output_dir":            "conditional_seg2mri",
        "prediction_type":       "v",
        "beta_schedule":         "cosine",
        "attention_resolutions": (8, 4),
        "cond_channels":         1,
        "num_size_classes":      0,
    },
    "novel_segmentation": {
        "output_dir":            "novel_segmentation",
        "prediction_type":       "v",
        "beta_schedule":         "cosine",
        "attention_resolutions": (8, 4),
        "cond_channels":         0,
        "num_size_classes":      3,
    },
}

spec = MODELS[MODEL]
print(f"loaded spec for {MODEL}: {spec}")


In [ ]:
def load_model(spec):
    cfg = Config(
        image_size=IMAGE_SIZE,
        attention_resolutions=spec["attention_resolutions"],
        cond_channels=spec["cond_channels"],
        num_size_classes=spec["num_size_classes"],
        prediction_type=spec["prediction_type"],
        beta_schedule=spec["beta_schedule"],
    )
    model = UNet3D(cfg).to(device)

    ckpt_path = CHECKPOINT_ROOT / spec["output_dir"] / "checkpoints" / "final_model.pt"
    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])

    # apply EMA shadow (the actual trained-on weights)
    if "ema" in ckpt and "shadow" in ckpt["ema"]:
        for name, p in model.named_parameters():
            if name in ckpt["ema"]["shadow"]:
                p.data.copy_(ckpt["ema"]["shadow"][name].to(device))

    model.eval()
    return model, ckpt_path


model, ckpt_path = load_model(spec)
print(f"loaded {MODEL} from {ckpt_path}")


In [ ]:
# Prepare the conditioning for each of N samples.
# - seg2mri: load N random seg masks from disk
# - novel_segmentation: cycle through the size classes (0, 1, 2)

if MODEL == "seg2mri":
    seg_files = sorted(SEG_DIR.glob("*.nii.gz"))
    assert len(seg_files) > 0, f"no seg files in {SEG_DIR}"

    rng = np.random.default_rng(SEED_BASE)
    chosen = rng.choice(len(seg_files), size=N_SAMPLES, replace=True)

    def get_cond_kwargs(i):
        seg_path = seg_files[chosen[i]]
        seg = nib.load(str(seg_path)).get_fdata().astype(np.float32)
        # same {0..3} -> [-1, 1] normalization as BraTSCondDataset
        seg = seg / (spec.get("num_seg_classes", 4) - 1) * 2.0 - 1.0
        seg = torch.from_numpy(seg).unsqueeze(0).unsqueeze(0).to(device)
        return {"cond": seg}, seg_path.stem

elif MODEL == "novel_segmentation":
    K = spec["num_size_classes"]

    def get_cond_kwargs(i):
        cls = i % K   # cycle 0, 1, 2, 0, 1, 2, ...
        size_cond = torch.tensor([cls], device=device, dtype=torch.long)
        return {"size_cond": size_cond}, f"class{cls}"


In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
out_dir = OUTPUT_ROOT / f"samples_{MODEL}"
(out_dir / "samples").mkdir(parents=True, exist_ok=True)

scheduler = make_scheduler(SAMPLER, spec["beta_schedule"], spec["prediction_type"])

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

per_sample = []
for i in tqdm(range(N_SAMPLES), desc=f"gen {MODEL}"):
    cond_kwargs, cond_label = get_cond_kwargs(i)
    seed = SEED_BASE + i
    t0 = time.time()
    vol = sample(model, scheduler, IMAGE_SIZE,
                 num_steps=SAMPLER_STEPS, device=device, seed=seed,
                 guidance_scale=GUIDANCE_SCALE, **cond_kwargs)
    dt = time.time() - t0

    arr = vol.squeeze().cpu().numpy().astype(np.float32)
    out_path = out_dir / "samples" / f"sample_{i+1:04d}.nii.gz"
    nib.save(nib.Nifti1Image(arr, np.eye(4)), str(out_path))

    per_sample.append({
        "filename":   out_path.name,
        "cond":       cond_label,
        "seed":       seed,
        "time_sec":   round(dt, 3),
        "voxel_min":  round(float(arr.min()), 4),
        "voxel_max":  round(float(arr.max()), 4),
        "voxel_mean": round(float(arr.mean()), 4),
        "voxel_std":  round(float(arr.std()), 4),
    })

peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2) if device == "cuda" else None
with (out_dir / "stats.json").open("w") as f:
    json.dump({
        "model":           MODEL,
        "sampler":         SAMPLER,
        "steps":           SAMPLER_STEPS,
        "guidance_scale":  GUIDANCE_SCALE,
        "n_samples":       N_SAMPLES,
        "checkpoint":      str(ckpt_path),
        "peak_gpu_gb":     peak_gb,
        "per_sample":      per_sample,
    }, f, indent=2)

print(f"\ndone. {N_SAMPLES} samples + stats.json -> {out_dir}")
